# Tri-ARC Kaggle Evaluation Notebook

This notebook bootstraps the offline Tri-ARC environment, executes the full ARC-AGI-2 evaluation split, and emits a competition-ready `submission.json`. Attach the following Kaggle datasets before running:
1. **tri-arc-code** – repository snapshot mounted at `/kaggle/input/tri-arc/Tri-ARC`.
2. **tri-arc-wheels** – prebuilt Python wheel cache located under `vendor/wheels/`.
3. **tri-arc-data** – ARC-AGI-2 datasets mounted alongside the repo (already referenced by `data/arcagi2/`).

All cells are idempotent; re-run individual stages as needed if the session times out.

## 1. Copy repository into the writable workspace

Kaggle mounts datasets as read-only. We copy the Tri-ARC repo into `/kaggle/working/` so the environment setup and evaluation run can write checkpoints, logs, and outputs.

In [ ]:
import os
import shutil
from pathlib import Path

SOURCE = Path("/kaggle/input/tri-arc/Tri-ARC")
TARGET = Path("/kaggle/working/Tri-ARC")

if not SOURCE.exists():
    raise FileNotFoundError(f"Source repository missing: {SOURCE}")

if TARGET.exists():
    print("Tri-ARC repository already present in /kaggle/working.")
else:
    print("Copying repository into writable workspace...")
    shutil.copytree(SOURCE, TARGET)

print("Workspace ready:", TARGET)
print("Sample entries:", sorted(p.name for p in TARGET.iterdir())[:10])

## 2. Configure environment variables

Logs are redirected to `/kaggle/temp/` to avoid quota limits on `/kaggle/working/`. These variables propagate to the shell commands invoked later in the notebook.

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("/kaggle/working/Tri-ARC")
TEMP_LOGS = Path("/kaggle/temp/Tri-ARC/logs")
TEMP_LOGS.mkdir(parents=True, exist_ok=True)

os.environ["TRI_ARC_ROOT"] = str(BASE_DIR)
os.environ["TRI_ARC_DATA_ROOT"] = str(BASE_DIR / "data" / "arcagi2")
os.environ["TRI_ARC_CACHE_ROOT"] = str(BASE_DIR / "cache")
os.environ["TRI_ARC_LOG_ROOT"] = str(TEMP_LOGS)
os.environ["PYTHONPATH"] = str(BASE_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TRI_ARC_RUN_ID"] = os.environ.get("TRI_ARC_RUN_ID", "kaggle_pass1")

for key in ["TRI_ARC_ROOT", "TRI_ARC_DATA_ROOT", "TRI_ARC_LOG_ROOT", "CUDA_VISIBLE_DEVICES", "TRI_ARC_RUN_ID"]:
    print(f"{key} = {os.environ[key]}")

## 3. Bootstrap the offline Python environment

The `scripts/setup_kaggle_env.sh` script creates a virtual environment and installs all dependencies from the bundled wheel cache. This step takes a few minutes on first run.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
chmod +x scripts/setup_kaggle_env.sh
bash scripts/setup_kaggle_env.sh
source .venv/bin/activate
python --version
pip --version

## 4. Run the full evaluation split

This cell launches the orchestrator with all solver modules enabled. Adjust `TRI_ARC_RUN_ID` if you need to differentiate multiple attempts.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
source .venv/bin/activate
export CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-0,1,2,3}
RUN_ID=${TRI_ARC_RUN_ID:-kaggle_pass1}
python scripts/run_eval.py --preset evaluation --split evaluation --run-id ${RUN_ID} --log-level INFO

## 5. Inspect run artifacts

Quick sanity check to confirm that metrics, runtime logs, and summaries exist before producing the submission file.

In [ ]:
import json
import os
from pathlib import Path

run_id = os.environ.get("TRI_ARC_RUN_ID", "kaggle_pass1")
run_root = Path("/kaggle/working/Tri-ARC/reports/eval_runs/full") / run_id
summary_path = run_root / "summary.json"
metrics_path = run_root / "metrics.jsonl"
runtime_log = run_root / "logs" / "runtime.log"

print("Run directory:", run_root)
for path in [summary_path, metrics_path, runtime_log]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2)[:1000])

## 6. Build `submission.json`

The helper script consolidates `outputs/pass2.json` into the two-attempt-per-output format required by Kaggle.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
source .venv/bin/activate
python tools/make_submission.py --output submission.json
ls -lh submission.json outputs/pass2.json

## 7. Package artifacts for download / dataset upload

Bundle the key outputs (submission file, pass2 archive, logs, and reports) into a single tarball. Use the Kaggle “Download Output” button to retrieve it once the notebook finishes.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
RUN_ID=${TRI_ARC_RUN_ID:-kaggle_pass1}
tar -czf tri_arc_artifacts.tar.gz \
  submission.json \
  outputs/pass2.json \
  reports/eval_runs/full/${RUN_ID}
ls -lh tri_arc_artifacts.tar.gz